# KGC Adaptation Thesis — Kaggle runner

**Settings:** Accelerator **GPU T4 ×2** · Internet **ON** · Persistence **Variables and Files**
**Add Input:** your `kgc-data` dataset.

> ⚠️ T4 is Turing (SM 7.5) → **fp16 only, no bf16**. Config already set.
> ⚠️ For training use **Save & Run All (Commit)** — interactive sessions die on disconnect.


## 0 · Clone + install

In [ ]:
REPO = "https://github.com/USERNAME/kgc-adaptation-thesis.git"   # <-- edit
PRIVATE = False

import os, subprocess, sys
if PRIVATE:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    REPO = REPO.replace("https://", f"https://{tok}@")

if not os.path.exists("/kaggle/working/repo"):
    subprocess.run(["git","clone",REPO,"/kaggle/working/repo"], check=True)
else:
    subprocess.run(["git","-C","/kaggle/working/repo","pull"], check=True)

os.chdir("/kaggle/working/repo")
sys.path.insert(0, "/kaggle/working/repo")
print(subprocess.run(["git","log","-1","--oneline"], capture_output=True, text=True).stdout)

In [ ]:
!pip install -q -r requirements.txt

# MoRA is a FORK of peft and must be installed LAST -- it overwrites official peft.
# If it breaks BOFT, fall back to LoRA vs BOFT (still two distinct mechanisms).
INSTALL_MORA = True
if INSTALL_MORA:
    !pip install -q git+https://github.com/kongds/MoRA.git#subdirectory=peft-mora

## 1 · Attach data
Symlink the Kaggle Dataset so `data/` resolves without copying.

In [ ]:
import os, glob
SRC = "/kaggle/input/kgc-data"          # <-- your dataset slug
os.makedirs("data", exist_ok=True)
for d in glob.glob(f"{SRC}/*"):
    dst = os.path.join("data", os.path.basename(d))
    if not os.path.exists(dst):
        os.symlink(d, dst)
print(sorted(os.listdir("data")))

## 2 · Smoke test — 15 min
Catches API breakage, fp16 instability and the `peft-mora` conflict **before** you spend a 12-hour session.

In [ ]:
!python -m scripts.smoke_test

## 3 · Build instruction data
10,000 triples → ~20,000 instances (KG-LLM emits 1 positive + 1 negative each). Stratified by relation so rare relations survive.

In [ ]:
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42
!python -m src.data.build_instructions --dataset WN11 --n_triples 10000 --seed 42 --anonymise

## 4 · Chapter 1 — format vs knowledge
**Two training runs.** Everything else is inference over the same outputs.
Binary task (WN11/FB13) → chance = 50, which is what makes KG-LLM's untuned 21.1 / 9.1 unambiguous.

In [ ]:
!python -m chapters.ch1_diagnostic.run --dataset WN11
!python -m chapters.ch1_diagnostic.run --dataset WN11 --anonymise

In [ ]:
!python -m chapters.ch1_diagnostic.analyse --dataset WN11 --smi

## 5 · Chapter 2 — the |E| sweep
★ **Two T4s = two jobs at once.** This is the biggest throughput lever you have.

In [ ]:
!python -m chapters.ch2_adaptation.run --sweep

In [ ]:
import subprocess
def pair(a, b):
    pa = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=0 {a}", shell=True)
    pb = subprocess.Popen(f"CUDA_VISIBLE_DEVICES=1 {b}", shell=True)
    return pa.wait(), pb.wait()

BASE = "python -m chapters.ch2_adaptation.run --dataset YAGO3-10 --triples 10000"
for E in (10000, 25000, 50000, 123182):
    print(f"===== |E| = {E:,} =====")
    pair(f"{BASE} --peft lora --entities {E}", f"{BASE} --peft mora --entities {E}")

In [ ]:
# BOFT (preservation hypothesis) + frozen probe (the CONTROL that makes a flat
# MoRA result interpretable). Probe does no training.
pair("python -m chapters.ch2_adaptation.run --peft boft  --entities 123182",
     "python -m chapters.ch2_adaptation.run --peft probe --entities 123182")

### 5b · DPO — Phase 2, on the winner only
KG-LLM trains on `random.choice(all_entities)`. We use type-consistent near-misses.

In [ ]:
SFT = "checkpoints/ch2-mora-E123182-T10000-s42"     # <-- the Phase-1 winner
!python -m chapters.ch2_adaptation.run --peft dpo --sft-adapter {SFT} \
        --negatives type_consistent --entities 123182 --triples 10000

In [ ]:
!python -m chapters.ch2_adaptation.analyse --forgetting

## 6 · Chapter 3 — the conditioning ladder
Routing analysis + faithfulness cost **no training**. Run `--analyse` first and check the `rich` band: if it is ~0%, the ∅ branch never fires and a 0% skip rate is a data problem, not a result.

In [ ]:
!python -m chapters.ch3_conditioning.run --dataset YAGO3-10 --analyse

In [ ]:
for LEVEL in ["L0","L1","L2","L3"]:          # L4 is the first thing to cut
    !python -m chapters.ch3_conditioning.run --dataset YAGO3-10 --level {LEVEL} --train

## 7 · Chapter 4 — measurement
★ **ZERO training.** Inference over checkpoints from Chapters 2–3, which is why this chapter is the safest and holds the verified 0/188 gaps.

In [ ]:
ADAPTER = "checkpoints/ch2-mora-E123182-T10000-s42"
!python -m chapters.ch4_measurement.run --adapter {ADAPTER} --dataset YAGO3-10 --limit 2000

## 8 · Package results
Download `results.zip` from the notebook Output tab, then run the app locally.

In [ ]:
!zip -qr /kaggle/working/results.zip results/
!du -sh /kaggle/working/results.zip
!ls -la checkpoints/ 2>/dev/null | head -20

---
### Session notes
* **12-hour cap** — checkpointing is on (`save_steps: 250`); resume from `checkpoints/<run>/checkpoint-*`
* **~30 GPU-h/week**, roughly doubled by running two jobs on the two T4s
* **Adapters are 20–100 MB** (adapter-only checkpointing) — safe for `/kaggle/working`
* **Download `results/` every session** — `/kaggle/working` is not permanent
